#### BERTopic
1. 임베딩
2. 차원축소(PCA)
3. 군집(Kmean)
4. 대표어 추출


#### c-TF-IDF
- 관점을 바꿧다 -> 한 토픽에 속한 문서 전부를 하나의 큰 문서처럼 합친 후 , 그 토픽을 다른 토픽과 구별해주는 단어를 뽑는 방식
- 예) 
    - 반도체 토픽에서 삼성, 반도체, 수출 -> 자주 나오고, 다른 토픽에서 잘 안나온다면 
    - 이 단어가 그 토픽의 대표어가 된다.

In [1]:
import pandas as pd
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "dragonkue/BGE-m3-ko",
    device="cpu"
)

data = pd.read_csv("../data/11-1_뉴스정제.csv")
data.head(1)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

,제목,본문,카테고리,요약,출처URL,정제본문
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,경제,"6 6일 현대백화점그룹이 광주시에 문화복합몰을 만든다고 6일 밝혔으며, 광주시는 서...",https://n.news.naver.com/mnews/article/001/001...,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...


In [2]:
from bertopic.backend import BaseEmbedder

In [3]:

class E5KoreanEmbedder(BaseEmbedder):
    def __init__(self):
        super().__init__()
        self.model = SentenceTransformer(
            "dragonkue/BGE-m3-ko",
            device="cpu")
    def embed(self, texts, verbose=False):
        return self.model.encode(
            texts,
            batch_size=32, normalize_embeddings=True, show_progress_bar=False,
        )

embedder = E5KoreanEmbedder()
print("임베더 준비 완료")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

임베더 준비 완료


In [4]:
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import CountVectorizer

In [5]:
kiwi = Kiwi()

def tokenize_with_kiwi(text):
    tokens = kiwi.tokenize(text)

    return [
        token.form
        for token in tokens
        if token.tag in keep_pos
        and len(token.form) > 1
        and token.form not in stop_words
    ]

# 가져올 명사, 동사, 형용사, 외국어 태그를 지정
keep_pos = {"NNG","NNP","VV","VA","SL"}
stop_words = {"기자","뉴스","사진","제공","관련","대해","통해","위해"}


In [6]:
def kiwi_tokenize(text):
    return [t.form for t in kiwi_tokenize(text)
            if t.tag in keep_pos and len(t.form) > 1 and t.form not in stop_words]

vectorizer = CountVectorizer(
    tokenizer=kiwi_tokenize, token_pattern=None, lowercase=False,
    ngram_range=(1, 2), min_df=2, max_df=0.9,
)

### 차원 축소,  군집 설정
- UMAP
- HDBSCAN

In [7]:
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                metric="cosine", random_state=42, low_memory=True)
hdbscan_model = HDBSCAN(min_cluster_size=10, min_samples=5, metric="euclidean",
                     cluster_selection_method="eom", prediction_data=False)
print("UMAP·HDBSCAN 준비 완료")

UMAP·HDBSCAN 준비 완료


In [8]:
docs = [d.strip() for d in data["정제본문"].tolist() if isinstance(d, str) and d.strip()][:100]

In [10]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

topic_model = BERTopic(
    embedding_model=embedder,
    vectorizer_model=vectorizer,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    representation_model=KeyBERTInspired(),
    calculate_probabilities=False,
    top_n_words=8,
    verbose=False,
)

topics, _ = topic_model.fit_transform(docs)
print("학습 완료. 문서마다 토픽 번호가 매겨졌습니다.")

KeyboardInterrupt: 

In [ ]:
info = topic_model.get_topic_info()

In [ ]:
info[["Topic","Count","Name"]].head()

In [ ]:
real_topic = [t for t in info['Topic']if t != -1]

NameError: name 'topic_model' is not defined